### Creation agent avec model llama-3.1-8b-instant (groq provider):

In [2]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from dotenv.ipython import load_dotenv

In [3]:
load_dotenv(override=True)

True

In [4]:
llm = ChatGroq(model="llama-3.1-8b-instant",temperature=0)

In [5]:
agent = create_agent(
model=llm,
system_prompt="You are a helpful assistant"
)

In [6]:
resp = agent.invoke(input={"messages":[
    {"role":"user", "content":"Je m'appel Taoufik KEHAL"}
]})

In [7]:
print(resp['messages'][-1].content)

Bonjour Taoufik KEHAL, je suis ravi de faire votre connaissance. Je suis là pour vous aider et répondre à vos questions. Qu'est-ce que je peux faire pour vous aujourd'hui ?


In [8]:
resp = agent.invoke(input={"messages":[
    {"role":"user", "content":"Comment je m'appel"}
]})

In [9]:
print(resp['messages'][-1].content)

Je suis ravi de vous rencontrer ! Je suis un assistant virtuel, donc je n'ai pas de nom personnel, mais je suis là pour vous aider et répondre à vos questions. Qu'est-ce que vous voulez savoir ou discuter ?


### Selection dynamique du modele a utiliser:

In [10]:
from langchain.agents.middleware import ModelRequest,ModelResponse,wrap_model_call
from langchain.messages import HumanMessage, SystemMessage, AIMessage

In [11]:
basic_llm =ChatGroq(model="meta-llama/llama-prompt-guard-2-86m",temperature=0)
advanced_llm = ChatGroq(model="llama-3.1-8b-instant",temperature=0)

In [12]:
@wrap_model_call
def dynamic_model_selection(request:ModelRequest, hundler) -> ModelResponse:

    env = request.runtime.context.get("env", "test")
    if env == "prod":
        model =advanced_llm
        print("advanced llm selected")
    else:
    
        model = basic_llm
        print("basic llm selected")

    return hundler(request.override(model=model))

In [13]:
agent2 = create_agent(
model=basic_llm,
tools=[],
middleware=[dynamic_model_selection],
debug =True
)

In [14]:
resp = agent2.invoke(
    input={"messages":[HumanMessage("c'est auoi un agent AI")]},
    context={"env":"prod"})

[values] {'messages': [HumanMessage(content="c'est auoi un agent AI", additional_kwargs={}, response_metadata={}, id='780410cc-88a0-4c2b-83d2-809d42dabb6d')]}
advanced llm selected
[updates] {'model': {'messages': [AIMessage(content="Bonjour ! Je suis un agent AI, également appelé chatbot ou modèle de langage artificiel. Je suis conçu pour comprendre et répondre à vos questions, discuter avec vous sur divers sujets, et même vous aider à résoudre des problèmes.\n\nJe suis basé sur des algorithmes de machine learning qui me permettent d'apprendre et d'améliorer mes capacités de compréhension et de réponse au fil du temps. Je peux comprendre et générer du texte en français, ainsi que d'autres langues.\n\nJe suis là pour vous aider, répondre à vos questions, et discuter avec vous sur des sujets tels que :\n\n- La science et la technologie\n- L'histoire et la culture\n- La littérature et l'art\n- La musique et le cinéma\n- Les actualités et les événements du monde\n- Et bien d'autres choses

In [15]:
from IPython.display import Markdown

In [16]:
Markdown(resp["messages"][-1].content)

Bonjour ! Je suis un agent AI, également appelé chatbot ou modèle de langage artificiel. Je suis conçu pour comprendre et répondre à vos questions, discuter avec vous sur divers sujets, et même vous aider à résoudre des problèmes.

Je suis basé sur des algorithmes de machine learning qui me permettent d'apprendre et d'améliorer mes capacités de compréhension et de réponse au fil du temps. Je peux comprendre et générer du texte en français, ainsi que d'autres langues.

Je suis là pour vous aider, répondre à vos questions, et discuter avec vous sur des sujets tels que :

- La science et la technologie
- L'histoire et la culture
- La littérature et l'art
- La musique et le cinéma
- Les actualités et les événements du monde
- Et bien d'autres choses encore !

Qu'est-ce que vous voulez discuter aujourd'hui ?

### Ajouter de la memoire a l'agent:

In [17]:

agent = create_agent(
model=llm,
system_prompt="You are a helpful assistant"

)

In [18]:
resp =agent.invoke(input={"messages":[HumanMessage("Je m'appelle Taoufik")]})



In [19]:
print(resp['messages'][-1].content)

Enchanté, Taoufik ! Je suis ravi de faire votre connaissance. Comment puis-je vous aider aujourd'hui ?


In [20]:
resp =agent.invoke(input={"messages":[HumanMessage("C'est quoi mon nom")]})


In [21]:
print(resp['messages'][-1].content)

Je suis désolé, mais je ne connais pas votre nom. Nous venons de commencer notre conversation, et je n'ai aucune information sur vous. Si vous voulez partager votre nom avec moi, je serais ravi de le connaître !


In [22]:
from langgraph.checkpoint.memory import InMemorySaver

In [23]:
memory = InMemorySaver()
agent = create_agent(
model=llm,
system_prompt="You are a helpful assistant",
checkpointer= memory
)

In [24]:
config= {"configurable":{"thread_id":1}}
resp =agent.invoke(input={"messages":[HumanMessage("Je m'appelle Taoufik")]},
                   config = config)


In [25]:
print(resp['messages'][-1].content)

Enchanté, Taoufik ! Je suis ravi de faire votre connaissance. Comment puis-je vous aider aujourd'hui ?


In [26]:
config= {"configurable":{"thread_id":1}}
resp =agent.invoke(input={"messages":[HumanMessage("Comment je m''appelle")]},
                   config = config)

In [27]:
print(resp['messages'][-1].content)

Vous vous appelez Taoufik ! C'est un beau prénom arabe, n'est-ce pas ? Qu'est-ce que vous aimez faire, Taoufik ?


### Ajouter des outils a l'agent:

In [28]:
from langchain.tools import tool

In [29]:
@tool
def get_weather(city: str):
    """
    Get the wather of the given city
    """
    print("weather Tool invoked")
    return {"city":city,
            "temperature":23,
            "humidity":80}

In [30]:
@tool
def get_employee_info(employe_name: str):
    """
    Get infos about the given employee (salary, seniority)
    """
    print("get_employee_info Tool invoked")
    return {"name":employe_name,
            "salary":340000,
            "seniority":8}

In [31]:
agent4 = create_agent(
    model=llm,
    tools=[get_weather, get_employee_info],
    checkpointer=memory,
    system_prompt="answer user qustion using provided tools"
)

In [32]:
resp = agent4.invoke(input={"messages":HumanMessage("La meteo a casablanca")}, config = config)

weather Tool invoked


In [33]:
print(resp['messages'][-1].content)

La météo à Casablanca est chaude et humide avec une température de 23 degrés et une humidité de 80%.


In [34]:
resp = agent4.invoke(input={"messages":HumanMessage("Les informations de l'empoyee Taoufik")}, config = config)

get_employee_info Tool invoked


In [35]:
print(resp['messages'][-1].content)

Les informations de l'employé Taoufik sont les suivantes : nom : Taoufik, salaire : 340 000, ancienneté : 8 ans.


Ajouter web search tool

In [36]:
from langchain_tavily import TavilySearch

In [37]:
tavily =TavilySearch(maxresults =10, search_depth="advanced")

In [38]:
@tool
def web_search(query:str, maxresult =10):
    """
    Searche fo general current web results
    """
    print("search web invoked")
    results = tavily.invoke({"query":query})
    
    return results

In [39]:
agent5 = create_agent(
    model=llm,
    tools=[get_weather, get_employee_info,web_search],
    system_prompt="answer user qustion using provided tools"
)

In [40]:
resp =agent5.invoke(input={"messages":HumanMessage("Actualites sur le maroc")},)

search web invoked


In [41]:
print(resp["messages"][-1].content)

Il semble que les actualités au Maroc soient variées et concernent divers domaines tels que la politique, l'économie, la culture et les sports. Voici quelques-unes des actualités les plus importantes :

* Le roi Mohammed VI a nommé le prince héritier Moulay El Hassan comme coordinateur des bureaux et services de l'état-major général des Forces armées royales.
* Le Maroc s'impose désormais comme le leader incontesté des exportations de tomates vers l'Europe.
* La Chine applique une exonération totale des droits de douane sur ses importations depuis l'Afrique, avec la première cargaison à bénéficier de cette mesure étant un lot de gypse importé du Maroc.
* Le Consulat général des États-Unis a inauguré son nouveau siège à Casablanca Finance City en présence de hautes personnalités marocaines et américaines.
* Le gouvernement a acté la réduction du temps de travail à 8 heures pour les agents de sécurité.

Il est important de noter que ces actualités sont sujettes à changement et peuvent êt

Ajouter tool pour executer du code python

In [44]:
from langchain_experimental.tools import PythonREPLTool

In [48]:
repl_tool = PythonREPLTool(sanitize_input=False)

In [60]:
agent = create_agent(
    model=llm,
    tools =[repl_tool],
    system_prompt="Generer et executer du code python pour repondre a la question de l'utilisateur, en placant le code python generer et le resultat dans le fichier doc.txt",
    debug =True
)

In [61]:
res = agent.invoke(input={"messages":[HumanMessage("Quel est la somme de 2 et 3 ?")]})

[values] {'messages': [HumanMessage(content='Quel est la somme de 2 et 3 ?', additional_kwargs={}, response_metadata={}, id='0bc23805-02ac-4212-bb98-44f7452afd6b')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'sj7vr1tq8', 'function': {'arguments': '{"query":"print(2 + 3)"}', 'name': 'Python_REPL'}, 'type': 'function'}, {'id': 'pb5djh1p9', 'function': {'arguments': '{"query":"with open(\'doc.txt\', \'w\') as f: f.write(\'La somme de 2 et 3 est : 5\')"}', 'name': 'Python_REPL'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 330, 'total_tokens': 394, 'completion_time': 0.08525919, 'completion_tokens_details': None, 'prompt_time': 0.020372626, 'prompt_tokens_details': None, 'queue_time': 0.04500527, 'total_time': 0.105631816}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d9492c3c54', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'mod

In [59]:
print(res["messages"][-1].content)

Le fichier dox.txt contient maintenant la phrase "La somme de 2 et 3 est : 5".


### Dynamique prompting

In [62]:
from typing import TypedDict
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt,ModelResponse

In [63]:
class Context(TypedDict):
    user_role:str

In [64]:
@dynamic_prompt
def user_role_prompt(request:ModelRequest) ->str:
    """
    Generate system prompt based on user role
    """
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant"
    if user_role == "expert":
        return f"{base_prompt} Provide detailed and technical answers to the user's questions"
    else:
        return f"{base_prompt} Provide simple and concise answers to the user's questions"
    
    return base_prompt

In [69]:
agent = create_agent(
    model=llm,
    tools= [],
    middleware=[user_role_prompt],
    debug =True)

In [70]:
result = agent.invoke(input={"messages":[HumanMessage("Explique moi le machine learning")]},
                      context={"user_role":"expert"})

[values] {'messages': [HumanMessage(content='Explique moi le machine learning', additional_kwargs={}, response_metadata={}, id='3d54ab10-5181-4f85-8655-d7cdcade994a')]}
[updates] {'model': {'messages': [AIMessage(content="Le machine learning (ML) est une sous-discipline de l'intelligence artificielle (IA) qui consiste à développer des algorithmes et des modèles mathématiques pour permettre aux ordinateurs de faire des prédictions, de prendre des décisions ou d'apprendre à partir de données.\n\n**Qu'est-ce que le machine learning ?**\n\nLe machine learning est un processus automatique qui permet aux ordinateurs de découvrir des modèles et des relations dans les données, sans avoir besoin d'être explicitement programmés pour cela. Les ordinateurs apprennent à partir de l'expérience et des données pour améliorer leurs performances et leurs capacités.\n\n**Types de machine learning**\n\nIl existe trois types principaux de machine learning :\n\n1. **Apprentissage supervisé** : dans ce type 

In [73]:
print(display(Markdown(result["messages"][-1].content)))

Le machine learning (ML) est une sous-discipline de l'intelligence artificielle (IA) qui consiste à développer des algorithmes et des modèles mathématiques pour permettre aux ordinateurs de faire des prédictions, de prendre des décisions ou d'apprendre à partir de données.

**Qu'est-ce que le machine learning ?**

Le machine learning est un processus automatique qui permet aux ordinateurs de découvrir des modèles et des relations dans les données, sans avoir besoin d'être explicitement programmés pour cela. Les ordinateurs apprennent à partir de l'expérience et des données pour améliorer leurs performances et leurs capacités.

**Types de machine learning**

Il existe trois types principaux de machine learning :

1. **Apprentissage supervisé** : dans ce type de machine learning, les données sont étiquetées et les algorithmes apprennent à prédire les résultats en fonction de ces étiquettes. Exemple : reconnaissance de chiffres, classification de textes.
2. **Apprentissage non supervisé** : dans ce type de machine learning, les données ne sont pas étiquetées et les algorithmes découvrent des modèles et des relations dans les données. Exemple : clustering, détection d'anomalies.
3. **Apprentissage par renforcement** : dans ce type de machine learning, les algorithmes apprennent à prendre des décisions en fonction des récompenses ou des pénalités qu'ils reçoivent. Exemple : jeux vidéo, robotique.

**Algorithmes de machine learning**

Certains des algorithmes de machine learning les plus courants incluent :

1. **Réseaux de neurones** : inspirés par le fonctionnement du cerveau humain, ces algorithmes utilisent des couches de neurones pour apprendre à reconnaître des modèles.
2. **Arbres de décision** : ces algorithmes utilisent des arbres pour prédire les résultats en fonction de critères spécifiques.
3. **SVM (Support Vector Machine)** : ces algorithmes utilisent des vecteurs de support pour trouver des frontières de séparation entre les classes.
4. **K-means** : ces algorithmes utilisent une méthode de clustering pour regrouper les données en classes similaires.

**Avantages du machine learning**

Le machine learning offre de nombreux avantages, notamment :

1. **Amélioration de la précision** : les algorithmes de machine learning peuvent apprendre à faire des prédictions plus précises que les algorithmes traditionnels.
2. **Réduction des coûts** : les algorithmes de machine learning peuvent automatiser certaines tâches, réduisant ainsi les coûts de main-d'œuvre.
3. **Amélioration de la rapidité** : les algorithmes de machine learning peuvent traiter de grandes quantités de données rapidement et efficacement.

**Défis du machine learning**

Le machine learning présente également certains défis, notamment :

1. **Qualité des données** : les algorithmes de machine learning nécessitent des données de haute qualité pour fonctionner correctement.
2. **Surapprentissage** : les algorithmes de machine learning peuvent surapprendre, ce qui signifie qu'ils apprennent à prédire les résultats de manière trop précise.
3. **Interprétabilité** : les algorithmes de machine learning peuvent être difficiles à interpréter, ce qui peut rendre difficile la compréhension de leurs décisions.

En résumé, le machine learning est une sous-discipline de l'intelligence artificielle qui consiste à développer des algorithmes et des modèles mathématiques pour permettre aux ordinateurs de faire des prédictions, de prendre des décisions ou d'apprendre à partir de données. Les algorithmes de machine learning peuvent offrir de nombreux avantages, mais ils présentent également certains défis qui doivent être pris en compte.

None


In [74]:
result = agent.invoke(input={"messages":[HumanMessage("Explique moi le machine learning")]},
                      context={"user_role":"beginner"})

[values] {'messages': [HumanMessage(content='Explique moi le machine learning', additional_kwargs={}, response_metadata={}, id='1d77a2ca-f302-414e-89dd-bf3383a18726')]}
[updates] {'model': {'messages': [AIMessage(content="Le machine learning (ML) est une branche de l'intelligence artificielle (IA) qui permet à un ordinateur de s'apprendre et d'améliorer ses performances sans être explicitement programmé.\n\nVoici les étapes clés du machine learning :\n\n1. **Collecte de données** : On collecte des données réelles ou simulées pour entraîner le modèle.\n2. **Prétraitement des données** : On nettoie, transforme et normalise les données pour les rendre utilisables.\n3. **Sélection du modèle** : On choisit un modèle de machine learning adapté au problème à résoudre.\n4. **Entraînement du modèle** : On entraîne le modèle avec les données collectées et prétraitées.\n5. **Évaluation du modèle** : On évalue les performances du modèle en utilisant des données de test.\n6. **Amélioration du modèl

In [75]:
print(display(Markdown(result["messages"][-1].content)))

Le machine learning (ML) est une branche de l'intelligence artificielle (IA) qui permet à un ordinateur de s'apprendre et d'améliorer ses performances sans être explicitement programmé.

Voici les étapes clés du machine learning :

1. **Collecte de données** : On collecte des données réelles ou simulées pour entraîner le modèle.
2. **Prétraitement des données** : On nettoie, transforme et normalise les données pour les rendre utilisables.
3. **Sélection du modèle** : On choisit un modèle de machine learning adapté au problème à résoudre.
4. **Entraînement du modèle** : On entraîne le modèle avec les données collectées et prétraitées.
5. **Évaluation du modèle** : On évalue les performances du modèle en utilisant des données de test.
6. **Amélioration du modèle** : On ajuste les paramètres du modèle pour améliorer ses performances.

Les types de machine learning les plus courants sont :

* **Apprentissage supervisé** : Le modèle est entraîné avec des données étiquetées pour prédire des résultats.
* **Apprentissage non supervisé** : Le modèle est entraîné avec des données non étiquetées pour découvrir des modèles ou des relations.
* **Apprentissage par renforcement** : Le modèle apprend à prendre des décisions en fonction des récompenses ou des pénalités.

Les applications du machine learning sont nombreuses, notamment :

* **Reconnaissance d'images** : Le modèle peut reconnaître des objets, des visages ou des scènes.
* **Analyse de texte** : Le modèle peut analyser et comprendre le sens des textes.
* **Prédiction** : Le modèle peut prédire des résultats ou des tendances.
* **Automatisation** : Le modèle peut automatiser des tâches répétitives ou complexes.

En résumé, le machine learning est une technique qui permet à un ordinateur de s'apprendre et d'améliorer ses performances en utilisant des données et des algorithmes.

None
